In [5]:
def my_decorator(original_func):
    # This is the "Bucket" (lives in the defaults of 'wrapper')
    def wrapper(*args, count_bucket=[0]):
        # This is the "Backpack" -> it holds 'original_func'
        count_bucket[0] += 1
        print(f"Call count: {count_bucket[0]}")
        return original_func(*args)
    
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")


In [6]:
say_hello()

Call count: 1
Hello!


In [7]:
say_hello()

Call count: 2
Hello!


In [8]:
def my_decorator(original_func):
    count = 0  # This is the "State" we want to remember
    
    def wrapper(*args, **kwargs):
        nonlocal count  # THE KEY: "Reach into the backpack and let me edit 'count'"
        count += 1
        print(f"I've been called {count} times!")
        return original_func(*args, **kwargs)
    
    return wrapper

@my_decorator
def greet():
    print("Hello!")

greet() # I've been called 1 times!
greet() # I've been called 2 times!


I've been called 1 times!
Hello!
I've been called 2 times!
Hello!


The "Nonlocal" way essentially turns a variable in the Backpack into a Bucket by giving you permission to change it.

In our analogy, here is the breakdown of who is who:
1. The Backpack = The Closure
This is the "Hidden Storage" that the inner function carries around.
What lives here: Any variable defined in the outer function (like name or count).
The Logic: These variables are "packed away" when the inner function is created. They stay alive as long as that function exists.
2. The Bucket = The Mutable Default Argument (The [] or {})
I called the default list a "bucket" because it is a single physical container in memory that never gets replaced.
What lives here: Data that you want to persist and easily change without using special keywords like nonlocal.
The Logic: Since the function always points to the same bucket in memory, you can keep throwing things into it (appending) and they will stay there forever.
Summary of the difference:
The Backpack is where Python stores Variables from the outside world.
The Bucket is a Specific Object (like a list) that the function "owns" as a default setting.
The "Nonlocal" way essentially turns a variable in the Backpack into a Bucket by giving you permission to change it.

In [9]:
def hello():
  print("Heyyy there!!")
  
hello()
greet = hello
greet()

Heyyy there!!
Heyyy there!!


In [10]:
del hello

In [11]:
hello()

NameError: name 'hello' is not defined

In [ ]:
greet()

Heyyy there!!


In [12]:
def cool():
    def super_cool():
        return 'I am very cool!'
    print(f"id for super cool function inside cool fn is :{id(super_cool)}")
    return super_cool

# 1. First Run
some_func = cool()
id_first = id(some_func)
print(f"First object ID:  {id_first}")

# 2. Second Run (Re-assigning to the same variable)
some_func = cool()
id_second = id(some_func)
print(f"Second object ID: {id_second}")

# 3. Verification
if id_first != id_second:
    print("\nSUCCESS: The first object was replaced by a brand new one in a different memory location!")


id for super cool function inside cool fn is :1974218783760
First object ID:  1974218783760
id for super cool function inside cool fn is :1974218784112
Second object ID: 1974218784112

SUCCESS: The first object was replaced by a brand new one in a different memory location!


In [13]:
def cool():
    count = 0
    def super_cool():
        nonlocal count  # Permission to change the backpack data
        count += 1
        return count
    return super_cool

counter = cool()
print(counter()) # 1
print(counter()) # 2 (It remembers and updates!)


1
2


In [14]:
counter()

3

In [15]:
counter()

4

In [16]:
# Assuming you ran the counter code above:
cell = counter.__closure__[0]
print(cell.cell_contents) # Output: 2


4


In [17]:
def cool(name):
    age = 25  # Not used in inner, will be DELETED
    def super_cool():
        # 'name' is used here, so it's kept in the "backpack"
        return f"Hello {name}!" 
    return super_cool

my_func = cool("Jatin")


In [18]:
my_func()


'Hello Jatin!'

In [19]:
def cool(name):
    age = 25  # Not used in inner, will be DELETED
    def super_cool():
        # 'name' is used here, so it's kept in the "backpack"
        return f"Hello {name}!" 
    return super_cool

cool("Jassi Singh")()

'Hello Jassi Singh!'

In [20]:
def cool(name):
    ageDefault = 25  
    def super_cool(age=ageDefault):
        # 'name' is used here, so it's kept in the "backpack"
        return f"Hello {name}! of age :{age}" 
    return super_cool

print(cool("Jassi Singh")(34))
print(cool("Jatin Singh")(26))

Hello Jassi Singh! of age :34
Hello Jatin Singh! of age :26


In this specific code, ageDefault is not saved in the "backpack" (closure). Instead, its value is hard-coded into the super_cool function object at the moment it is created.
Here is exactly what is happening in memory:
1. Default Arguments vs. Closures
The Default Argument (age=ageDefault): When Python defines super_cool, it looks up the value of ageDefault (which is 25) and stores that actual number in a special attribute of the function called __defaults__. It essentially "bakes" the number 25 into the function.
The Closure (name): Because name is used inside the body of the function, Python keeps the variable name alive in the __closure__ (the backpack).
2. Is ageDefault kept alive?
No. After cool("Jassi Singh") finishes its run:
The value 25 is already safe inside the super_cool function's default settings.
The variable name ageDefault itself is deleted from memory.
If you tried to look for ageDefault later, you wouldn't find it.
3. How to Prove it
You can inspect the function object to see where the data is stored:

In [21]:
func = cool("Jassi Singh")

# Look at the "Backpack" (Closure) -> Only 'name' is here
print(f"Closure contents: {[c.cell_contents for c in func.__closure__]}") 
# Output: ['Jassi Singh']

# Look at the Default Arguments -> The number 25 is stored here
print(f"Default arguments: {func.__defaults__}") 
# Output: (25,)


Closure contents: ['Jassi Singh']
Default arguments: (25,)


In [22]:
def add_to_list(item, my_list=[]): # The trap is here!
    my_list.append(item)
    return my_list

print(add_to_list("A")) # Output: ['A']
print(add_to_list("B")) # Output: ['A', 'B']  <-- WAIT, WHAT?
print(add_to_list("C")) # Output: ['A', 'B', 'C']


['A']
['A', 'B']
['A', 'B', 'C']


In [23]:
def add_to_list(item, my_list=None):
    if my_list is None:
        my_list = [] # A brand NEW list is created every time the function runs
    my_list.append(item)
    return my_list

print(add_to_list("A")) # ['A']
print(add_to_list("B")) # ['B']  <-- Fixed!


['A']
['B']


In [25]:
def cool(name):
    # This list is created every time cool() is called
    my_defaults = [] 
    # default argument trap
    def super_cool(items=my_defaults):
        items.append(name)
        return items
    
    return super_cool

# Case 1
f1 = cool("Jatin")
print(f1())  # ['Jatin']
print(f1())  # ['Jatin', 'Jatin'] -> IT PERSISTS for f1!

# Case 2
f2 = cool("Singh")
print(f2())  # ['Singh'] -> IT IS A FRESH LIST for f2!

print(f1())
print(f2())

print(f"f1 defaults: {f1.__defaults__}")
print(f"f2 defaults: {f2.__defaults__}")
print(f"backpack of f1: {[cell.cell_contents for cell in f1.__closure__]}")
print(f"backpack of f2: {[cell.cell_contents for cell in f2.__closure__]}")
print(f"f1 closure: {f1.__closure__}")
print(f"f1 closure's cell: {f1.__closure__[0]}")
print(f"f1 closure's cell's content : {f1.__closure__[0].cell_contents}")

['Jatin']
['Jatin', 'Jatin']
['Singh']
['Jatin', 'Jatin', 'Jatin']
['Singh', 'Singh']
f1 defaults: (['Jatin', 'Jatin', 'Jatin'],)
f2 defaults: (['Singh', 'Singh'],)
backpack of f1: ['Jatin']
backpack of f2: ['Singh']
f1 closure: (<cell at 0x000001CBA8430610: str object at 0x000001CBA8570FC0>,)
f1 closure's cell: <cell at 0x000001CBA8430610: str object at 0x000001CBA8570FC0>
f1 closure's cell's content : Jatin
